## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

## 2. Load Processed Data

In [ ]:
# low_memory = False suppresses DtypeWarning on mixed-type columns
games_details = pd.read_csv(
    "../data/processed/games_details_cleaned.csv",
    low_memory=False
)

In [ ]:
games = pd.read_csv(
    "../data/processed/games_with_teams.csv",
    parse_dates=["GAME_DATE_EST"]
)
games.head()

In [ ]:
ranking = pd.read_csv("../data/processed/ranking_cleaned.csv")

## 3. Select Required Columns

In [ ]:
games_clean = games[[
    "GAME_DATE_EST",
    "GAME_ID",
    "SEASON",
    "HOME_TEAM_NAME",
    "HOME_CITY",
    "AWAY_TEAM_NAME",
    "AWAY_CITY",
    "PTS_home",
    "PTS_away",
    "FG_PCT_home",
    "FG_PCT_away",
    "AST_home",
    "AST_away",
    "REB_home",
    "REB_away",
    "HOME_TEAM_WINS",
    "GAME_RESULT",
    "TOTAL_POINTS"
]].copy()

In [ ]:
ranking_clean = ranking[[
    "TEAM_ID",
    "SEASON_ID",
    "CONFERENCE"
]].copy()

## 4. Feature Engineering

In [ ]:
# Point differential (home perspective)
games_clean["POINT_DIFF"] = games_clean["PTS_home"] - games_clean["PTS_away"]

In [ ]:
# Winning team name
games_clean["WINNING_TEAM"] = np.where(
    games_clean["HOME_TEAM_WINS"] == 1,
    games_clean["HOME_TEAM_NAME"],
    games_clean["AWAY_TEAM_NAME"]
)

In [ ]:
# Calendar year of the game
games_clean["YEAR"] = games_clean["GAME_DATE_EST"].dt.year

In [ ]:
# Binary win flags
games_clean["HOME_WIN"] = games_clean["HOME_TEAM_WINS"]
games_clean["AWAY_WIN"] = 1 - games_clean["HOME_TEAM_WINS"]

## 5. Aggregations

In [ ]:
# Home vs Away win rate and scoring by season
home_away_summary = games_clean.groupby("SEASON").agg(
    HOME_WIN_RATE=("HOME_WIN", "mean"),
    AVG_HOME_POINTS=("PTS_home", "mean"),
    AVG_AWAY_POINTS=("PTS_away", "mean")
).reset_index()

home_away_summary.head()

In [ ]:
# Per-team per-season home game stats
team_stats = games_clean.groupby(["SEASON", "HOME_TEAM_NAME"]).agg(
    GAMES_PLAYED=("GAME_ID", "count"),
    AVG_POINTS_SCORED=("PTS_home", "mean"),
    AVG_POINTS_CONCEDED=("PTS_away", "mean"),
    WIN_RATE=("HOME_TEAM_WINS", "mean")
).reset_index()

team_stats.head()

In [ ]:
# Season-level summary
season_summary = games_clean.groupby("SEASON").agg(
    AVG_POINTS=("TOTAL_POINTS", "mean"),
    HOME_WIN_RATE=("HOME_TEAM_WINS", "mean"),
    TOTAL_GAMES=("GAME_ID", "count")
).reset_index()

season_summary.head()

## 6. Player Summary

In [ ]:
games_details.info()

In [ ]:
# Attach SEASON to player box scores via GAME_ID
games_details = games_details.merge(
    games_clean[["GAME_ID", "SEASON"]],
    on="GAME_ID",
    how="left"
)

In [ ]:
# Per-player per-season averages (min 20 games played)
player_summary = games_details.groupby(
    ["SEASON", "Player"]
).agg(
    AVG_POINTS=("Points", "mean"),
    AVG_REB=("REB", "mean"),
    AVG_AST=("AST", "mean"),
    GAMES_PLAYED=("GAME_ID", "count")
).reset_index()

player_summary = player_summary[player_summary["GAMES_PLAYED"] >= 20]
player_summary.head()

## 7. Export All Analytical Tables

In [ ]:
games_clean.to_csv("../data/processed/games_features.csv", index=False)
team_stats.to_csv("../data/processed/team_stats.csv", index=False)
home_away_summary.to_csv("../data/processed/home_away_summary.csv", index=False)
season_summary.to_csv("../data/processed/season_summary.csv", index=False)
player_summary.to_csv("../data/processed/player_summary.csv", index=False)

print("All feature tables exported to data/processed/")